In [11]:
!pip install --user regex sentence-transformers
!pip install pgvector
!pip install psycopg2-binary
!pip install datasets



[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
from sentence_transformers import SentenceTransformer 
import numpy as np
import sqlite3

DB_FILE = "court.db"

name = 'deepvk/USER2-small'
model = SentenceTransformer(name)


Loading weights: 100%|██████████| 74/74 [00:00<00:00, 38022.60it/s]


In [13]:
def init_db():
    conn = sqlite3.connect(DB_FILE)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS court_documents (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            case_number TEXT,
            content TEXT,
            embedding BLOB NOT NULL,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    """)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS document_relations (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            doc_id_1 INTEGER NOT NULL,
            doc_id_2 INTEGER NOT NULL,
            similarity REAL NOT NULL,
            relation_type TEXT DEFAULT 'duplicate',
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            UNIQUE(doc_id_1, doc_id_2),
            FOREIGN KEY (doc_id_1) REFERENCES court_documents(id) ON DELETE CASCADE,
            FOREIGN KEY (doc_id_2) REFERENCES court_documents(id) ON DELETE CASCADE
        )
    """)
    conn.execute("""
        CREATE INDEX IF NOT EXISTS idx_relations_doc1
        ON document_relations(doc_id_1)
    """)
    conn.execute("""
        CREATE INDEX IF NOT EXISTS idx_relations_doc2
        ON document_relations(doc_id_2)
    """)
    conn.commit()
    conn.close()
    print("База готова.")

In [14]:
def get_embedding(text: str) -> np.ndarray:
    medium = model.encode(
        [f"search_document: {text}"],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0].astype(np.float32)
    return medium

In [15]:
def doc_relations(case_number: str, content: str, threshold=0.85, top=20):
    medium = get_embedding(content)
    medium_bytes = medium.tobytes()

    conn = sqlite3.connect(DB_FILE)
    curs = conn.cursor()

    curs.execute( # вставка нового
        "INSERT INTO court_documents (case_number, content, embedding) "
        "VALUES (?, ?, ?)",
        (case_number, content, medium_bytes)
    )
    new_id = curs.lastrowid 

    # загрузка остальных
    rows = curs.execute(
        "SELECT id, embedding FROM court_documents WHERE id != ?",
        (new_id,)
    ).fetchall()

    if rows:
        ids = [r[0] for r in rows]
        matrix = np.vstack([np.frombuffer(r[1], dtype=np.float32) for r in rows])
        sims = matrix @ medium
        closest = np.argsort(sims)[::-1][:top]

        for idx in closest:
            sim_value = float(sims[idx])
            if sim_value >= threshold:
                a, b = sorted([new_id, ids[idx]])
                curs.execute(
                    """
                    INSERT INTO document_relations
                        (doc_id_1, doc_id_2, similarity, relation_type)
                    VALUES (?, ?, ?, 'duplicate')
                    ON CONFLICT(doc_id_1, doc_id_2) DO UPDATE
                    SET similarity = MAX(document_relations.similarity, excluded.similarity)
                    """,
                    (a, b, sim_value)
                )
    conn.commit()
    conn.close()
    return new_id


In [16]:
from sklearn.metrics.pairwise import cosine_similarity

def duplicats(doc_id: int, threshold=0.85):
    conn = sqlite3.connect(DB_FILE)
    rows = conn.execute(
        """
        SELECT
            CASE WHEN doc_id_1 = ? THEN doc_id_2 ELSE doc_id_1 END AS related_id,
            similarity,
            relation_type
        FROM document_relations
        WHERE (doc_id_1 = ? OR doc_id_2 = ?)
          AND similarity >= ?
        ORDER BY similarity DESC
        """,
        (doc_id, doc_id, doc_id, threshold)
    ).fetchall()
    conn.close()
    return [
        {'related_id': r[0], 'similarity': r[1], 'type': r[2]}
        for r in rows
    ]

In [17]:
def old(threshold=0.85, top=20):
    conn = sqlite3.connect(DB_FILE, timeout=10)
    cur = conn.cursor()

    rows = cur.execute(
        "SELECT id, embedding FROM court_documents WHERE embedding IS NOT NULL"
    ).fetchall()
    ids = [r[0] for r in rows]
    matrix = np.vstack([np.frombuffer(r[1], dtype=np.float32) for r in rows])

    for i, doc_id in enumerate(ids):
        sims = matrix @ matrix[i]
        order = np.argsort(sims)[::-1]
        count = 0
        for idx in order:
            if ids[idx] == doc_id:
                continue
            sim = float(sims[idx])
            if sim < threshold:
                continue
            a, b = sorted([doc_id, ids[idx]])
            cur.execute(
                """
                INSERT INTO document_relations
                    (doc_id_1, doc_id_2, similarity, relation_type)
                VALUES (?, ?, ?, 'duplicate')
                ON CONFLICT(doc_id_1, doc_id_2) DO UPDATE
                SET similarity = MAX(document_relations.similarity, excluded.similarity)
                """,
                (a, b, sim)
            )
            count += 1
            if count >= top:
                break
    conn.commit()
    print(f"Обработано документов: {len(ids)}")

In [18]:

init_db()

База готова.


In [19]:
# --- Запуск: наполняем базу из датасета ---
init_db()

from datasets import load_dataset

ds = load_dataset("TryDotAtwo/russian-legal-ner", split="train")
print(ds.features)

for i, row in enumerate(ds):
    text = row['text']
    case_number = f"dataset_row_{i}"
    doc_relations(case_number, text)
    print(f"Обработан документ {i}, id в базе получен")

База готова.


C:\Users\User\AppData\Roaming\Python\Python314\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\User\.cache\huggingface\hub\datasets--TryDotAtwo--russian-legal-ner. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
C:\Users\User\AppData\Roaming\Python\Python314\site-packages\huggingface_hub\file_download.py:14

{'text': Value('string'), 'spans': Value('string')}
Обработан документ 0, id в базе получен
Обработан документ 1, id в базе получен
Обработан документ 2, id в базе получен
Обработан документ 3, id в базе получен
Обработан документ 4, id в базе получен
Обработан документ 5, id в базе получен
Обработан документ 6, id в базе получен
Обработан документ 7, id в базе получен
Обработан документ 8, id в базе получен
Обработан документ 9, id в базе получен
Обработан документ 10, id в базе получен
Обработан документ 11, id в базе получен
Обработан документ 12, id в базе получен
Обработан документ 13, id в базе получен
Обработан документ 14, id в базе получен
Обработан документ 15, id в базе получен
Обработан документ 16, id в базе получен
Обработан документ 17, id в базе получен
Обработан документ 18, id в базе получен
Обработан документ 19, id в базе получен
Обработан документ 20, id в базе получен
Обработан документ 21, id в базе получен
Обработан документ 22, id в базе получен
Обработан докум

KeyboardInterrupt: 

In [20]:
new_text = """
Истец обратился в суд с требованием о взыскании задолженности 
по договору займа в размере 75 000 рублей. Ответчик в судебное 
заседание не явился, о месте и времени рассмотрения дела извещён надлежащим образом.
"""

new_id = doc_relations(
    case_number="A40-999/2024",
    content=new_text
)

print(f"Новый документ сохранён с id={new_id}")

# Теперь смотрим, с чем он связался
links = duplicats(new_id, threshold=0.80)

if links:
    print("\nНайдены связи:")
    for link in links:
        print(f"  -> id={link['related_id']}, сходство={link['similarity']:.4f}, тип={link['type']}")
else:
    print("\nСвязей не найдено — документ уникален.")

OperationalError: database is locked